# DRM · Lab 03 — Assignment 02 Sheet (GRADED)
## Forwards and Futures on **BANKNIFTY** and **Silver**

**Course:** DRM (<ECON F354>) · <Semester 1 - 2026-27>
**Instructor:** <Prof. Aditya Sharma> · BITS Pilani
**Total marks:** 45 **Deadline:** `<18 Oct 2026, Midnight>`

---

### Before you start

1. Work through `Lab03_Example_Forwards_and_Futures.ipynb` first. Every technique needed here
   appears there. You are expected to adapt it, not to invent from scratch.
2. Fill in the identification cell below. **A submission without it will not be graded.**
3. Write your code in the cells marked `# ---- YOUR CODE HERE ----` and your written answers in
   the markdown cells marked `**Your answer:**`. Do not delete the question cells.
4. **Marks are for interpretation as much as for code.** A correct plot with no commentary earns
   roughly half the marks for that part. Every question that asks "why" or "comment" is looking
   for economics, not for a restatement of what the chart shows.

### Submission

- `Kernel → Restart & Run All`, confirm there are **no errors and all outputs are visible**.
- Save as **`DRM_Lab03_<CampusID>.ipynb`** — e.g. `DRM_Lab03_2023A7PS0123P.ipynb`.
- Upload to the Lab 03 assignment on **Nalanda**. Do not email it; do not open a pull request.

### Marks

| Q | Topic | Marks |
|---|---|---|
| Q1 | Data acquisition and exploratory analysis | 5 |
| Q2 | Silver forward curve from cost of carry | 5 |
| Q3 | BANKNIFTY theoretical futures and the expiry calendar | 5 |
| Q4 | Back-test: theoretical vs market curve | 5 |
| Q5 | Simulation: risk-neutral pricing and the basis process | 5 |
| Q6 | Short answers | 10 |
| Q7 | Application: hedging a bank portfolio | 10 |
| | **Total** | **100** |

In [ ]:
# ==================================================================================
#  IDENTIFICATION — fill this in. Submissions without it will not be graded.
# ==================================================================================
NAME       = ""          # e.g. "Ananya Sharma"
CAMPUS_ID  = ""          # e.g. "2023A7PS0123P"
EMAIL      = ""

# Honour statement. Type "I agree" to confirm that the work below is your own and that any
# outside help, source or AI assistance is disclosed in the final cell of this notebook.
HONOUR     = ""

assert NAME and CAMPUS_ID and HONOUR.strip().lower() == "i agree", \
    "Fill in NAME, CAMPUS_ID and type 'I agree' in HONOUR before running the rest."
print(f"{NAME} | {CAMPUS_ID}")

---
## Provided setup — run these two cells as-is

The data layer is given to you so that a network failure never blocks the assignment. It tries the
live API first and falls back to a seeded simulator, exactly as in the worked example. **If your
loaders report `[SIM ]`, say so in your write-up** and interpret your results accordingly — you
will not be penalised for working offline, but you will be penalised for presenting simulated
output as though it were market data.

In [ ]:
# !pip install -q yfinance pandas_datareader        # uncomment in Colab if needed

import warnings, datetime as dt
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

warnings.filterwarnings("ignore")
plt.rcParams.update({
    "figure.figsize": (11, 4.4), "figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25,
    "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 10, "axes.titlesize": 12, "axes.titleweight": "bold", "legend.frameon": False,
})
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

SEED, START, END = 2026, "2021-01-01", dt.date.today().isoformat()
DATA_MODE = "auto"          # "auto" | "live" | "offline"
IS_SIM = {}

def ttl(title, key=None):
    return title + ("   —   [SIMULATED DATA]" if IS_SIM.get(key) else "")

print(f"pandas {pd.__version__} | window {START} -> {END} | DATA_MODE = {DATA_MODE}")

In [ ]:
def fetch_yahoo(ticker, start=START, end=END):
    import yfinance as yf
    df = yf.download(ticker, start=start, end=end, progress=False, auto_adjust=False)
    if df is None or len(df) == 0:
        raise RuntimeError("empty frame")
    if isinstance(df.columns, pd.MultiIndex):
        df = df.droplevel(1, axis=1)
    s = df["Close"].astype(float).dropna()
    s.index = pd.to_datetime(s.index).tz_localize(None)
    return s.rename(ticker)

def fetch_fred(series, start=START, end=END):
    from pandas_datareader import data as pdr
    s = pdr.DataReader(series, "fred", start, end).iloc[:, 0].astype(float).dropna()
    s.index = pd.to_datetime(s.index)
    return s.rename(series)

def sim_gbm_series(name, S0, mu, sigma, seed, start=START, end=END):
    idx = pd.bdate_range(start, end)
    rng = np.random.default_rng(seed)
    dtt = 1 / 252
    incr = (mu - 0.5 * sigma**2) * dtt + sigma * np.sqrt(dtt) * rng.standard_normal(len(idx))
    return pd.Series(S0 * np.exp(np.cumsum(incr)), index=idx, name=name)

def sim_ou_series(name, x0, level, kappa, vol, seed, start=START, end=END):
    idx = pd.bdate_range(start, end)
    rng = np.random.default_rng(seed)
    dtt, x, out = 1 / 252, x0, np.empty(len(idx))
    for i in range(len(idx)):
        x += kappa * (level - x) * dtt + vol * np.sqrt(dtt) * rng.standard_normal()
        out[i] = x
    return pd.Series(out, index=idx, name=name)

def load(key, live_fn, sim_fn):
    if DATA_MODE != "offline":
        try:
            s = live_fn(); IS_SIM[key] = False
            print(f"  [LIVE] {key:<10s} {len(s):>5d} obs  {s.index.min().date()} -> {s.index.max().date()}")
            return s
        except Exception as e:
            if DATA_MODE == "live":
                raise
            print(f"  [SIM ] {key:<10s} live fetch failed ({type(e).__name__}) -> simulated")
    else:
        print(f"  [SIM ] {key:<10s} DATA_MODE=offline")
    IS_SIM[key] = True
    return sim_fn()

print("Loading\n" + "-" * 70)
banknifty = load("banknifty", lambda: fetch_yahoo("^NSEBANK"),
                              lambda: sim_gbm_series("^NSEBANK", 31_000, 0.11, 0.200, SEED + 71))
silver    = load("silver",    lambda: fetch_yahoo("SI=F"),
                              lambda: sim_gbm_series("SI=F", 26.5, 0.10, 0.280, 2100))
usd_r     = load("usd_r",     lambda: fetch_yahoo("^IRX") / 100.0,
                              lambda: sim_ou_series("USD 3M", 0.001, 0.045, 1.2, 0.010, SEED + 73))
inr_r     = load("inr_r",     lambda: fetch_fred("INDIRLTLT01STM") / 100.0,
                              lambda: sim_ou_series("INR 3M", 0.055, 0.066, 1.0, 0.007, SEED + 74))
print("-" * 70)

usd_r = usd_r.reindex(silver.index).ffill().bfill()
inr_r = inr_r.reindex(banknifty.index).ffill().bfill()
print(f"BANKNIFTY {banknifty.iloc[-1]:,.2f} | Silver {silver.iloc[-1]:,.3f} "
      f"| r_INR {inr_r.iloc[-1]:.3%} | r_USD {usd_r.iloc[-1]:.3%}")

---
## Q1 · Data acquisition and exploratory analysis · **[5 marks]**

**(a) [4]** Produce a 2×2 figure showing BANKNIFTY, silver, and both interest-rate series over the
full sample. Label axes with units.

**(b) [4]** Build a summary table with one row per price series containing: number of observations,
start and end dates, total return, **annualised volatility** (from daily log returns, $\sigma\sqrt{252}$),
annualised mean log return, and **maximum drawdown**.

**(c) [2]** In the markdown cell below, comment on the volatility figures. Silver is materially
more volatile than an equity index — state the ratio you obtain and give an economic reason.
Then answer: **does this volatility difference change the forward price of either asset? Justify
in one sentence.**

In [ ]:
# ---- YOUR CODE HERE ----  Q1 (a) and (b)

def max_drawdown(series):
    '''Return the maximum peak-to-trough decline as a NEGATIVE fraction, e.g. -0.32.'''
    # TODO
    raise NotImplementedError

def summary_stats(series):
    '''Return a dict with n_obs, start, end, total_return, ann_vol, ann_drift, max_dd.'''
    # TODO
    raise NotImplementedError

# TODO: build the 2x2 figure, then assemble and display the summary table.

**Your answer (Q1c):**

*Volatility ratio silver : BANKNIFTY = ______*

*Economic reason:*

*Effect on the forward price:*

---
## Q2 · Silver forward curve from cost of carry · **[5 marks]**

Silver is a commodity, so use the commodity form:

$$F_0(T) = S_0\,e^{(r + u - y)T}$$

**(a) [3]** Implement `forward_price(S, r, T, q=0, u=0, y=0)` and
`implied_convenience_yield(F, S, r, T, u=0)`. **Include an assertion that the two functions invert
each other** — a numerical check, not a comment.

**(b) [5]** Using the latest silver spot and USD short rate, plot the forward curve for maturities
1, 2, 3, 6, 9, 12, 18, 24 months under **four** convenience-yield scenarios of your choosing.
Alongside it, plot the same curves as **annualised carry**, $\frac{1}{T}\ln(F/S)$ in bps.

**(c) [3]** Silver's storage cost per dollar of value is *higher* than gold's — the same value takes
up far more vault space. Choose a value for $u$, and **justify it with a cited source** (an LBMA or
exchange storage schedule, a vaulting provider's published rate, or a broker's fee card). One line
of justification, with the source named, is enough.

**(d) [4]** State the break-even convenience yield at which the curve flips from contango to
backwardation, verify it numerically from your own curve, and comment on how plausible sustained
backwardation is for silver relative to gold. Which of the two is more likely to backwardate, and
why?

In [ ]:
# ---- YOUR CODE HERE ----  Q2 (a)

def forward_price(S, r, T, q=0.0, u=0.0, y=0.0):
    # TODO
    raise NotImplementedError

def implied_convenience_yield(F, S, r, T, u=0.0):
    # TODO
    raise NotImplementedError

# TODO: assert that implied_convenience_yield(forward_price(...)) recovers the y you put in.

In [ ]:
# ---- YOUR CODE HERE ----  Q2 (b)
# TODO: two-panel figure - forward curve (prices) and annualised carry (bps).

**Your answer (Q2c and Q2d):**

*Storage cost $u$ used, and source:*

*Break-even convenience yield (formula and numerical value):*

*Silver vs gold — which backwardates more readily, and why:*

---
## Q3 · BANKNIFTY theoretical futures and the expiry calendar · **[5 marks]**

**(a) [5]** Implement the expiry calendar.

> ⚠️ **NSE has changed the index-derivatives expiry weekday by circular more than once.** Do not
> hard-code a weekday from memory or from an old textbook. Look up the **currently applicable NSE
> circular**, cite it in your answer, and make the weekday a named constant so it can be audited.
> An incorrect expiry day silently corrupts every $\tau$ in Q4, and it is the single most common
> source of wrong answers in this lab.

Write `last_weekday_of_month(year, month, weekday)` and `next_expiries(asof, n)`. Verify your
function against three real expiry dates you look up independently.

**(b) [5]** Using the latest BANKNIFTY spot, the INR short rate, and a dividend yield $q$ that you
justify (BANKNIFTY's constituents are banks — look up the index dividend yield rather than reusing
Nifty 50's), compute the theoretical futures price for the **near, next and far** month contracts.
Report a table with expiry date, days to expiry, $\tau$, $F^{\text{theo}}$, basis in points and
basis in bps.

**(c) [5]** Obtain an **observed** BANKNIFTY futures price for the near-month contract — from the
NSE website, your broker's terminal, or an API — and record the timestamp. Compute the implied
financing rate $r_{\text{implied}} = q + \frac{1}{\tau}\ln(F^{\text{obs}}/S)$ and compare it with
the money-market rate. Comment on the gap.

In [ ]:
# ---- YOUR CODE HERE ----  Q3 (a)

NSE_EXPIRY_WEEKDAY = None      # TODO: set from the CURRENT NSE circular. Mon=0, Tue=1, ..., Thu=3
EXPIRY_SOURCE      = ""        # TODO: cite the circular / NSE page and the date you checked it

def last_weekday_of_month(year, month, weekday=NSE_EXPIRY_WEEKDAY):
    # TODO
    raise NotImplementedError

def next_expiries(asof, n=3, weekday=NSE_EXPIRY_WEEKDAY):
    # TODO
    raise NotImplementedError

# TODO: verify against three real expiry dates you looked up yourself.

In [ ]:
# ---- YOUR CODE HERE ----  Q3 (b) and (c)
Q_BANKNIFTY   = None      # TODO: dividend yield, with a source
F_OBS_NEAR    = None      # TODO: observed near-month futures price
F_OBS_ASOF    = ""        # TODO: timestamp of that quote, e.g. "2026-08-05 15:25 IST"
F_OBS_SOURCE  = ""        # TODO: where you got it

# TODO: table of near/next/far theoretical prices; then implied financing rate vs money-market rate.

**Your answer (Q3):**

*Expiry weekday used and the circular that establishes it:*

*Dividend yield used and source:*

*Implied financing rate vs money-market rate — size of the gap and your explanation:*

---
## Q4 · Back-test: theoretical vs market curve · **[5 marks]**

Rebuild the theoretical futures price on **every historical date**, using that date's spot, that
date's interest rate and the actual days remaining to that date's front contract, then compare with
the observed futures series.

**(a)** Build the panel with columns `S, r, expiry, dte, tau, F_theo, F_obs`. If you have a
real continuous front-month BANKNIFTY futures series (NSE bhavcopy, `jugaad-data`, `nsepython`),
use it and say so. If not, generate `F_obs` with a documented mean-reverting distortion of the
implied carry rate, as in the worked example — **and state clearly in your write-up that this half
of the comparison is synthetic.**

**(b)** Produce **three** diagnostic plots:
1. levels — spot, $F^{\text{theo}}$, $F^{\text{obs}}$ through time;
2. basis in bps through time, with a $\pm 25$ bps transaction-cost band marked;
3. **convergence** — mean $|{\rm basis}|$ against days to expiry.

**(c)** Report a statistics table: mean and standard deviation of the basis in bps, mean
$|{\rm basis}|$, the 95th percentile, the percentage of days on which $|{\rm basis}| > 25$ bps, and
the mean implied financing rate against the mean actual rate.

**(d)** Write **at least 150 words** answering all four of the following:

- Does the convergence plot behave as theory requires? If not, what is wrong with your data?
- Is the mean basis statistically and economically distinguishable from zero? If it is non-zero,
  give the **three most likely specification errors** in descending order of plausibility.
- After 25 bps of round-trip costs, is anything left? List at least three costs your calculation
  still ignores.
- The implied financing rate explodes in the last few days before expiry. Explain **mathematically**
  why, state the filter you applied, and justify the threshold you chose.

In [ ]:
# ---- YOUR CODE HERE ----  Q4 (a)

def build_panel(spot, rate, q, expiry_weekday=NSE_EXPIRY_WEEKDAY, seed=SEED + 81):
    # TODO: returns a DataFrame with S, r, expiry, dte, tau, F_theo, F_obs
    raise NotImplementedError

In [ ]:
# ---- YOUR CODE HERE ----  Q4 (b): the three diagnostic plots

In [ ]:
# ---- YOUR CODE HERE ----  Q4 (c): the statistics table
TC_BPS = 25.0

**Your answer (Q4d) — at least 150 words:**

*Convergence:*

*Mean basis and the three most likely specification errors:*

*After costs, and what is still ignored:*

*Why the implied rate explodes near expiry, and the filter you used:*

---
## Q5 · Simulation · **[5 marks]**

**(a)** Simulate BANKNIFTY under the **risk-neutral** measure,
$dS = (r-q)S\,dt + \sigma S\,dW^{\mathbb{Q}}$, and verify numerically that
$\mathbb{E}^{\mathbb{Q}}[S_T] = F_0(T)$ for $T = 0.25$. Produce a convergence table over
$N \in \{10^3, 5\times10^3, 2.5\times10^4, 10^5\}$ reporting the MC estimate, its standard error,
the closed-form value, the error in bps, **and the ratio error/SE**. State what the error/SE column
should look like if your estimator is unbiased, and confirm that yours does.

**(b)** Plot a fan chart of the simulated distribution with the forward curve overlaid. Report
$P(S_T > F_0(T))$ and explain why it is **not** 50%.

**(c)** For **silver**, simulate a two-factor model with a stochastic convenience yield:

$$dS = (r-\delta)S\,dt + \sigma_S S\,dW_1, \qquad
d\delta = \kappa(\alpha - \delta)dt + \sigma_\delta\,dW_2, \qquad
\text{corr}(dW_1,dW_2)=\rho$$

Price the curve **by Monte Carlo** as $F(0,T)=\mathbb{E}^{\mathbb{Q}}[S_T]$ — do not use a closed
form. Produce curves for at least three starting values $\delta_0$, including one that generates a
**backwardated front with a contango back**. State the half-life of $\delta$ implied by your $\kappa$
and explain how it determines where the curve pivots.

**(d)** Simulate the **basis** as a mean-reverting distortion of the implied carry rate and
compare it with your empirical basis from Q4 using: a density overlay, a Q-Q plot, and the mean,
standard deviation and excess kurtosis of both. **Moment-match the two samples first** (same mean
and standard deviation) so the comparison isolates distribution *shape*. Then answer: does a
Gaussian model reproduce the tails? If you run a KS test, does its verdict agree with what the Q-Q
plot shows — and if they disagree, which do you trust and why?

In [ ]:
# ---- YOUR CODE HERE ----  Q5 (a): Monte Carlo convergence table

In [ ]:
# ---- YOUR CODE HERE ----  Q5 (b): fan chart and P(S_T > F)

In [ ]:
# ---- YOUR CODE HERE ----  Q5 (c): two-factor silver model, curve by Monte Carlo

In [ ]:
# ---- YOUR CODE HERE ----  Q5 (d): simulated vs empirical basis

**Your answer (Q5):**

*(a) What error/SE should look like, and what you observe:*

*(b) $P(S_T > F)$ and why it is not 50%:*

*(c) Half-life of $\delta$, and how $\kappa$ controls the pivot point:*

*(d) Tails, and the KS test vs the Q-Q plot:*

---
## Q6 · Short answers · **[10 marks, 2 each]**

Answer in **two to four sentences each**. Marks are for the mechanism, not the assertion.

1. Silver has significant **industrial** demand; gold is overwhelmingly a monetary and jewellery
   asset. How should this difference show up in the two metals' convenience yields, and in the
   *shape and stability* of their forward curves?

2. An equity index futures contract trades **below** spot for several consecutive weeks. Give two
   distinct explanations — one that is consistent with no-arbitrage and one that involves a market
   friction — and describe an observation that would let you distinguish them.

3. Under **stochastic interest rates**, forward and futures prices on the same underlying and
   maturity are not equal. State which is larger when the underlying is **positively** correlated
   with interest rates, and explain the mechanism through the daily settlement of futures.

4. The volatility of the underlying does not appear in the cost-of-carry formula, yet it is the
   central input for an option on the same asset. Explain precisely why.

5. Suppose you observe a persistent 40 bps mispricing in BANKNIFTY futures that survives your
   transaction-cost band. Before concluding you have found an arbitrage, list **four** checks you
   would run, and say what each one would rule out.

**Your answers (Q6):**

1.

2.

3.

4.

5.

---
## Q7 · Application: hedging a bank portfolio · **[10 marks]**

You manage a ₹50 crore portfolio of Indian banking stocks. You want to hedge for the next month
using BANKNIFTY futures.

**(a) [4]** Estimate the **minimum-variance hedge ratio**

$$h^* = \rho\,\frac{\sigma_{\Delta S}}{\sigma_{\Delta F}}$$

from historical data. Use your Q4 panel for the futures series and construct a proxy portfolio
return series (you may use BANKNIFTY itself plus an idiosyncratic component you simulate, but
**state your choice and its consequence for the answer**).

**(b) [3]** Convert $h^*$ into a **number of contracts**, stating the BANKNIFTY contract multiplier
(lot size) you used and its source. Round appropriately and explain which way you rounded and why.

**(c) [3]** Report the variance reduction achieved, $1-\rho^2$ of residual variance, and identify
the **basis risk** that remains. Under what circumstance would this hedge perform noticeably worse
than your estimate suggests?

In [ ]:
# ---- YOUR CODE HERE ----  Q7
PORTFOLIO_VALUE   = 50e7      # ₹50 crore
BANKNIFTY_LOT     = None      # TODO: contract multiplier, with source
LOT_SOURCE        = ""

**Your answer (Q7):**

*Hedge ratio and how you constructed the portfolio series:*

*Number of contracts, lot size and source, rounding decision:*

*Variance reduction and the residual basis risk:*

---
## Declaration and checklist

**Sources, collaborators and any AI assistance used** (be specific — which tool, for which part,
and what you did with the output). Full Disclosure of the prompt is expected and costs you nothing; undisclosed use
of AI will be considered as an integrity violation.

*Your declaration:*

---

### Before you submit, confirm every box

- [ ] `NAME`, `CAMPUS_ID` and `HONOUR` are filled in and the identification cell runs
- [ ] `Kernel → Restart & Run All` completes with **no errors**
- [ ] Every plot and table is **visible in the saved file** (not an empty output cell)
- [ ] Every `**Your answer:**` cell is filled in — none left blank
- [ ] You stated, for each data series, whether it is **live or simulated**
- [ ] Sources cited for: NSE expiry circular, dividend yield, silver storage cost, lot size
- [ ] Saved as `DRM_Lab03_<CampusID>.ipynb`
- [ ] Uploaded to the Lab 03 assignment on **Nalanda** before the deadline